In [1]:
# Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Reading and Parsing JSON Files/Data")
    .master("local[*]")
    .getOrCreate()
)

spark

In [2]:
# Read Single line JSON file

df_single = spark.read.format("json").load("data/input/order_singleline.json")

In [3]:
df_single.printSchema()
df_single.show(truncate=False)

root
 |-- contact: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_line_items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- amount: double (nullable = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- qty: long (nullable = true)

+------------------------+-----------+--------+------------------------------------+
|contact                 |customer_id|order_id|order_line_items                    |
+------------------------+-----------+--------+------------------------------------+
|[9000010000, 9000010001]|C001       |O101    |[{102.45, I001, 6}, {2.01, I003, 2}]|
+------------------------+-----------+--------+------------------------------------+



In [4]:
# Read Multiline JSON file

df_multi = spark.read.format("json").option("multiLine", True).load("data/input/order_multiline.json")

# Setting multiLine=True allows Spark to read JSON where a single JSON object spans multiple lines.

In [5]:
df_multi.printSchema()
df_multi.show(truncate=False)

root
 |-- contact: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_line_items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- amount: double (nullable = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- qty: long (nullable = true)

+------------------------+-----------+--------+------------------------------------+
|contact                 |customer_id|order_id|order_line_items                    |
+------------------------+-----------+--------+------------------------------------+
|[9000010000, 9000010001]|C001       |O101    |[{102.45, I001, 6}, {2.01, I003, 2}]|
+------------------------+-----------+--------+------------------------------------+



In [6]:
df = spark.read.format("text").load("data/input/order_singleline.json")

In [7]:
df.printSchema()
df.show(truncate=False)

root
 |-- value: string (nullable = true)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                                              |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{"order_id":"O101","customer_id":"C001","order_line_items":[{"item_id":"I001","qty":6,"amount":102.45},{"item_id":"I003","qty":2,"amount":2.01}],"contact":[9000010000,9000010001]}|
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+



In [8]:
# With Schema

_schema = "customer_id string, order_id string, contact array<long>"

df_schema = spark.read.format("json").schema(_schema).load("data/input/order_singleline.json")

In [9]:
df_schema.show()

+-----------+--------+--------------------+
|customer_id|order_id|             contact|
+-----------+--------+--------------------+
|       C001|    O101|[9000010000, 9000...|
+-----------+--------+--------------------+



In [14]:
# root
#  |-- contact: array (nullable = true)
#  |    |-- element: long (containsNull = true)
#  |-- customer_id: string (nullable = true)
#  |-- order_id: string (nullable = true)
#  |-- order_line_items: array (nullable = true)
#  |    |-- element: struct (containsNull = true)
#  |    |    |-- amount: double (nullable = true)
#  |    |    |-- item_id: string (nullable = true)
#  |    |    |-- qty: long (nullable = true)

_schema = "contact array<long>, customer_id string, order_id string, order_line_items array<struct<amount double, item_id string, qty long>>"

# We can allso change datatype
_schema = "contact array<string>, customer_id string, order_id string, order_line_items array<struct<amount double, item_id string, qty long>>"

In [15]:
df_schema_new = spark.read.format("json").schema(_schema).load("data/input/order_singleline.json")

In [16]:
df_schema_new.printSchema()
df_schema_new.show(truncate=False)

root
 |-- contact: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_line_items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- amount: double (nullable = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- qty: long (nullable = true)

+------------------------+-----------+--------+------------------------------------+
|contact                 |customer_id|order_id|order_line_items                    |
+------------------------+-----------+--------+------------------------------------+
|[9000010000, 9000010001]|C001       |O101    |[{102.45, I001, 6}, {2.01, I003, 2}]|
+------------------------+-----------+--------+------------------------------------+



In [21]:
# Function from_json to read from a column

from pyspark.sql.functions import from_json

_schema = "contact array<string>, customer_id string, order_id string, order_line_items array<struct<amount double, item_id string, qty long>>"
df.show()
df_expanded = df.withColumn("parsed", from_json(df.value, _schema))
# df.value = JSON string column
# parsed = struct column with nested fields

+--------------------+
|               value|
+--------------------+
|{"order_id":"O101...|
+--------------------+



In [22]:
df_expanded.printSchema()
df_expanded.show(truncate=False)

root
 |-- value: string (nullable = true)
 |-- parsed: struct (nullable = true)
 |    |-- contact: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- customer_id: string (nullable = true)
 |    |-- order_id: string (nullable = true)
 |    |-- order_line_items: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- amount: double (nullable = true)
 |    |    |    |-- item_id: string (nullable = true)
 |    |    |    |-- qty: long (nullable = true)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------+
|value                                                                                                                                                                              |parsed                        

In [23]:
# Get values from Parsed JSON

df_final = df_expanded.select("parsed.*")
#df_final = df_expanded.select("parsed.contact")
df_final.show(truncate=False)

+------------------------+-----------+--------+------------------------------------+
|contact                 |customer_id|order_id|order_line_items                    |
+------------------------+-----------+--------+------------------------------------+
|[9000010000, 9000010001]|C001       |O101    |[{102.45, I001, 6}, {2.01, I003, 2}]|
+------------------------+-----------+--------+------------------------------------+



In [24]:
from pyspark.sql.functions import explode

df_final_2 = df_final.withColumn("expand_line_items", explode("order_line_items"))
df_final_2.show(truncate=False)
# Each element inside order_line_items becomes a separate row

df_final_3 = df_final.withColumn("expand_line_items", explode("order_line_items")).withColumn("expand_contact", explode("contact"))
df_final_3.show(truncate=False)

+------------------------+-----------+--------+------------------------------------+-----------------+
|contact                 |customer_id|order_id|order_line_items                    |expand_line_items|
+------------------------+-----------+--------+------------------------------------+-----------------+
|[9000010000, 9000010001]|C001       |O101    |[{102.45, I001, 6}, {2.01, I003, 2}]|{102.45, I001, 6}|
|[9000010000, 9000010001]|C001       |O101    |[{102.45, I001, 6}, {2.01, I003, 2}]|{2.01, I003, 2}  |
+------------------------+-----------+--------+------------------------------------+-----------------+

+------------------------+-----------+--------+------------------------------------+-----------------+--------------+
|contact                 |customer_id|order_id|order_line_items                    |expand_line_items|expand_contact|
+------------------------+-----------+--------+------------------------------------+-----------------+--------------+
|[9000010000, 9000010001]|C

In [25]:
df_select = df_final_2.select("contact", "customer_id", "order_id", "expand_line_items.*")
df_select.show()

#df_select_2 = df_final_3.select("expand_contact.*", "customer_id", "order_id", "expand_line_items.*")
# expand_contact is array<string> -> explode -> string, NOT a struct
# "expand_contact.*" -> invalid
# Only works for struct, not for primitive types like string

df_select_2 = df_final_3.select("expand_contact", "customer_id", "order_id", "expand_line_items.*")
#df_select_2 = df_final_3.select("expand_contact", "customer_id", "order_id", "expand_line_items")
df_select_2.show()

+--------------------+-----------+--------+------+-------+---+
|             contact|customer_id|order_id|amount|item_id|qty|
+--------------------+-----------+--------+------+-------+---+
|[9000010000, 9000...|       C001|    O101|102.45|   I001|  6|
|[9000010000, 9000...|       C001|    O101|  2.01|   I003|  2|
+--------------------+-----------+--------+------+-------+---+

+--------------+-----------+--------+------+-------+---+
|expand_contact|customer_id|order_id|amount|item_id|qty|
+--------------+-----------+--------+------+-------+---+
|    9000010000|       C001|    O101|102.45|   I001|  6|
|    9000010001|       C001|    O101|102.45|   I001|  6|
|    9000010000|       C001|    O101|  2.01|   I003|  2|
|    9000010001|       C001|    O101|  2.01|   I003|  2|
+--------------+-----------+--------+------+-------+---+



In [26]:
# Function to_json to parse a JSON string

from pyspark.sql.functions import to_json

df_unparsed = df_expanded.withColumn("unparsed", to_json(df_expanded.parsed))

In [27]:
df_unparsed.printSchema()
df_unparsed.select("unparsed").show(truncate=False)

root
 |-- value: string (nullable = true)
 |-- parsed: struct (nullable = true)
 |    |-- contact: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- customer_id: string (nullable = true)
 |    |-- order_id: string (nullable = true)
 |    |-- order_line_items: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- amount: double (nullable = true)
 |    |    |    |-- item_id: string (nullable = true)
 |    |    |    |-- qty: long (nullable = true)
 |-- unparsed: string (nullable = true)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|unparsed                                                                                                                                                                               |
+---------------------------------------------------------

In [28]:
# Explode Array fields
df_final = df_select.withColumn("contact_expanded", explode("contact"))

In [29]:
df_final.printSchema()

root
 |-- contact: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- item_id: string (nullable = true)
 |-- qty: long (nullable = true)
 |-- contact_expanded: string (nullable = true)



In [30]:
df_final.drop("contact").show()
# This does not modify df_final in place. To remove it permanently, we need to reassign it:
# df_final = df_final.drop("contact")

+-----------+--------+------+-------+---+----------------+
|customer_id|order_id|amount|item_id|qty|contact_expanded|
+-----------+--------+------+-------+---+----------------+
|       C001|    O101|102.45|   I001|  6|      9000010000|
|       C001|    O101|102.45|   I001|  6|      9000010001|
|       C001|    O101|  2.01|   I003|  2|      9000010000|
|       C001|    O101|  2.01|   I003|  2|      9000010001|
+-----------+--------+------+-------+---+----------------+



In [31]:
spark.stop()